# Demo 2: does spatial structure buy forecast skill?

Does a COVID-19 forecast improve when the model knows where people live and where they
travel? Demo 2 asks that of three models of the same three states (Massachusetts,
Connecticut, New Hampshire) over the 2020-21 season. They are identical except in their
geography and the movement between places:

| model | nodes | movement |
|---|---|---|
| `state` | 3 states | none: each state is one well-mixed population |
| `county` | 32 counties | none: 32 separate epidemics that share only their priors |
| `movement` | 32 counties | daily commuting between them, from the US Census |

Each is fitted to weekly hospital census and deaths with an ensemble Kalman filter, and
forecasts four weeks ahead from each of 24 weekly forecast dates. All three are scored at
the three states.

The notebook reads top to bottom:

0. Parameters
1. This introduction
2. Data: the geography, the season, population, commuting and the CDC observations
3. The compartment model
4. Parameters, priors and dynamics
5. The simplest model, `state`: one fit and one forecast, explained
6. The other two models, one change each
7. The rolling forecasts: every model at both seeds, then all three models' forecasts
   from the date of section 5

Everything that touches epymorph is in this notebook. Preparing data that is not
epymorph (reading the frozen CDC file, drawing initial conditions) is in
`covid_inputs.py`, and writing the runs to disk is in `runs.py`.

## 0. Parameters

Where the inputs are and where the runs go. A build overrides these with papermill; the
defaults work when the notebook is opened where it lives.

In [ ]:
data_dir = "../../../data"
runs_dir = "../../../results/demos/runs/covid"
output_dir = "../../../results/demos"
max_forecast_dates = None  # None for the manuscript; a small number makes a quick test

In [ ]:
import dataclasses
import time
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from epymorph.adrio.acs5 import Population
from epymorph.adrio.commuting_flows import Commuters
from epymorph.attribute import AttributeDef
from epymorph.compartment_model import CompartmentModel, compartment, edge, fork
from epymorph.data import mm
from epymorph.data_shape import Shapes
from epymorph.forecasting.dynamic_params import (
    ExponentialTransform,
    GaussianPrior,
    OrnsteinUhlenbeck,
)
from epymorph.forecasting.ensemble_kalman_filter import EnsembleKalmanFilterSimulator
from epymorph.forecasting.filter import ModelLink, Observations
from epymorph.forecasting.forecast import ForecastSimulator
from epymorph.forecasting.likelihood import PoissonLikelihood
from epymorph.forecasting.pipeline import PipelineConfig, UnknownParam
from epymorph.geography.us_census import CountyScope, StateScope
from epymorph.initializer import Explicit
from epymorph.rume import SingleStrataRUME
from epymorph.time import EveryNDays, TimeFrame
from joblib import Parallel, delayed
from sympy import Max

# The parts that are not epymorph: data preparation, writing runs, plotting.
from manuscript_analysis import covid_inputs, runs
from manuscript_analysis.data import ProvenanceCachedADRIO
from manuscript_analysis.demos.covid.evaluation import COLORS
from manuscript_analysis.demos.lib.figures import use_manuscript_style
from manuscript_analysis.demos.lib.plots import plot_single_forecast_grid
from manuscript_analysis.demos.lib.scoring import HUB_QUANTILE_LEVELS

use_manuscript_style()

## 2. Data

### Where and when

The geography is a **scope**: here the 32 counties of the three states and, for the
`state` model, the three states themselves, both as the Census drew them in 2015. The
**time frame** is the season: 35 weeks from 2020-08-09. The filter learns from the first
eight weeks before it forecasts anything, so the first forecast is made on 2020-10-04 and
the last on 2021-03-14.

The season opens on a Sunday because the two CDC series label each week at opposite ends:
the hospital data by the Sunday it starts, the deaths by the Saturday it ends.

In [ ]:
STATES = ["MA", "CT", "NH"]
CENSUS_YEAR = 2015
SEASON_START = date(2020, 8, 9)  # a Sunday
SEASON_WEEKS = 35
BURN_IN_WEEKS = 8  # assimilated before the first forecast
HORIZON_WEEKS = 4  # how far each forecast reaches

county_scope = CountyScope.in_states(STATES, year=CENSUS_YEAR)
state_scope = StateScope.in_states(STATES, year=CENSUS_YEAR)
season = TimeFrame.of(start_date=SEASON_START, duration_days=7 * SEASON_WEEKS)

counties = list(county_scope.node_ids)
states = list(state_scope.node_ids)
state_names = list(state_scope.labels)
print(f"{len(counties)} counties in {len(states)} states, {season.start_date} to {season.end_date}")

### Population and commuting

epymorph's **ADRIOs** fetch data for a scope and time frame: here the American Community
Survey's population and its county-to-county commuting flows. Each is wrapped in
`ProvenanceCachedADRIO`, which saves the result to `data/raw/` with a record of where and
when it came from the first time it is fetched, and reads that file ever after. So a build
needs no Census API key, and always gets the same numbers.

In [ ]:
def frozen(adrio, name, what):
    # The same ADRIO, read from its saved copy in data/raw/.
    return ProvenanceCachedADRIO(
        adrio,
        cache_path=Path(data_dir) / "raw" / f"covid-{name}-{CENSUS_YEAR}-{'-'.join(STATES).lower()}.npz",
        description=f"{what}; {CENSUS_YEAR} vintage; {', '.join(STATES)}",
    )


county_population = (
    frozen(Population(), "population-counties", "ACS 5-year population per county")
    .with_context(scope=county_scope, time_frame=season)
    .inspect().result.astype(np.int64)
)
state_population = (
    frozen(Population(), "population-states", "ACS 5-year population per state")
    .with_context(scope=state_scope, time_frame=season)
    .inspect().result.astype(np.int64)
)
commuters = np.asarray(
    frozen(Commuters(), "commuters-counties", "ACS commuting flows between counties")
    .with_context(scope=county_scope, time_frame=season)
    .inspect().result
).astype(np.int64)

crossing = (commuters.sum() - np.trace(commuters)) / commuters.sum()
print(f"{state_population.sum():,} people; {crossing:.0%} of workers commute across a county line")

### The observations

Two quantities, weekly and per county, from the CDC: the **hospital census**, the mean
number of COVID-19 inpatients on a day of that week (a stock), and **deaths**, the week's
count (a flow). CDC suppresses a county's weekly deaths when they are between 1 and 9; those
cells were imputed, by a random draw from the region's own death rate that week.

The observations were fetched once and frozen in `data/demos/covid/observations.csv`, so
the target every model is scored against cannot move, whatever happens to the sources.
(How they were fetched and imputed is in `scripts/fetch_covid_observations.py`, which no
build runs.)

In [ ]:
observed = covid_inputs.load_observations(
    Path(data_dir) / "demos" / "covid" / "observations.csv",
    counties, SEASON_START, SEASON_WEEKS,
)

# Each state's total, for the state model and for scoring.
county_to_state = covid_inputs.county_to_state(counties, states)  # (states, counties)
state_census = observed.census @ county_to_state.T
state_deaths = observed.deaths @ county_to_state.T

print(f"{observed.suppressed.mean():.0%} of county-weeks had their deaths suppressed")

fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.2), layout="constrained")
for s, name in enumerate(state_names):
    axs[0].plot(observed.census_dates, state_census[:, s], label=name)
    axs[1].plot(observed.death_dates, state_deaths[:, s], label=name)
axs[0].set_ylabel("hospital census")
axs[1].set_ylabel("weekly deaths")
for ax in axs:
    ax.tick_params(axis="x", labelrotation=45)
axs[0].legend()
plt.show()

## 3. The disease: an SEIRHD compartment model

Susceptible, exposed, infectious, recovered, hospitalized, dead. A compartment model is a
list of compartments, the parameters it needs, and the transitions between compartments as
symbolic rates. Deaths arrive only through hospital, so one model explains both quantities
we observe: the hospital census is the `H` compartment and deaths are the `H → D`
transition.

`fork` marks transitions that compete for the same people: someone leaving `I` is
hospitalized or recovers, in proportion to `hospitalization_prob`.

In [ ]:
class SEIRHD(CompartmentModel):
    compartments = (
        compartment("S"),
        compartment("E"),
        compartment("I"),
        compartment("R"),
        # Neither the hospitalized nor the dead commute.
        compartment("H", tags=["immobile"]),
        compartment("D", tags=["immobile"]),
    )

    requirements = (
        AttributeDef("beta", type=float, shape=Shapes.TxN, comment="infectivity"),
        AttributeDef("sigma", type=float, shape=Shapes.TxN,
                     comment="rate of leaving the latent period"),
        AttributeDef("gamma", type=float, shape=Shapes.TxN,
                     comment="rate of leaving the infectious period"),
        AttributeDef("xi", type=float, shape=Shapes.TxN, comment="immune waning rate"),
        AttributeDef("hospitalization_prob", type=float, shape=Shapes.TxN,
                     comment="probability of hospitalization given infection"),
        AttributeDef("hospitalization_duration", type=float, shape=Shapes.TxN,
                     comment="mean duration of hospitalization, in days"),
        AttributeDef("death_prob", type=float, shape=Shapes.TxN,
                     comment="deaths per hospitalization"),
    )

    def edges(self, symbols):
        [S, E, I, R, H, D] = symbols.all_compartments
        [beta, sigma, gamma, xi, h_prob, h_dur, d_prob] = symbols.all_requirements

        # The dead leave the mixing population; the hospitalized do not.
        N_alive = Max(1, S + E + I + R + H)

        return [
            edge(S, E, rate=beta * S * I / N_alive),
            edge(E, I, rate=sigma * E),
            fork(
                edge(I, H, rate=gamma * I * h_prob),
                edge(I, R, rate=gamma * I * (1 - h_prob)),
            ),
            fork(
                edge(H, D, rate=H / h_dur * d_prob),
                edge(H, R, rate=H / h_dur * (1 - d_prob)),
            ),
            edge(R, S, rate=xi * R),
        ]


# epymorph turns the definition back into readable rates.
for event in SEIRHD().events:
    print(f"{event.name!s:7} {event.rate}")

## 4. Parameters, priors and dynamics

Most parameters are fixed, at values from the literature or measured from the data.

In [ ]:
LATENT_DAYS = 3.0
INFECTIOUS_DAYS = 6.0
IMMUNE_DAYS = 365.0
HOSPITAL_DAYS = 7.0  # measured: the season's patient-days / admissions = 6.97
HOSPITALIZATION_PROB = 0.048  # per infection

Two are **estimated** by the filter, for each node and changing over time: the
transmission rate `beta` and the death probability `death_prob`. (`death_prob` is deaths
per hospitalization, but every death in this model passes through `H`, while the CDC
counts deaths outside hospital too; so it sits above any published in-hospital figure.)

Each is estimated on the log scale, so it stays positive, with:

- a **prior**: what we believe before any data, a normal distribution on the log scale;
- **dynamics**: how it may drift between observations, an Ornstein-Uhlenbeck process that
  wanders but is pulled back towards its mean.

`ExponentialTransform("log_beta")` is what connects the two: the model's `beta` is
`exp(log_beta)`, and `log_beta` is what the filter estimates.

In [ ]:
BETA = 0.20  # prior median; with a 6-day infectious period, R0 = 1.2
DEATH_PROB = 0.22  # prior median
log_beta = float(np.log(BETA))
log_death_prob = float(np.log(DEATH_PROB))

unknown_params = {
    "ipm::log_beta": UnknownParam(
        prior=GaussianPrior(mean=log_beta, standard_deviation=0.30),
        dynamics=OrnsteinUhlenbeck(damping=1 / 150, mean=log_beta, standard_deviation=0.45),
    ),
    "ipm::log_death_prob": UnknownParam(
        prior=GaussianPrior(mean=log_death_prob, standard_deviation=0.15),
        dynamics=OrnsteinUhlenbeck(damping=1 / 150, mean=log_death_prob, standard_deviation=0.15),
    ),
}

# Every parameter the compartment model requires, for the state model. The other two
# models change only the population, and the movement model adds commuting.
state_params = {
    "ipm::beta": ExponentialTransform("log_beta"),
    "ipm::sigma": 1 / LATENT_DAYS,
    "ipm::gamma": 1 / INFECTIOUS_DAYS,
    "ipm::xi": 1 / IMMUNE_DAYS,
    "ipm::hospitalization_prob": HOSPITALIZATION_PROB,
    "ipm::hospitalization_duration": HOSPITAL_DAYS,
    "ipm::death_prob": ExponentialTransform("log_death_prob"),
    "*::population": state_population,
}

The **initial conditions** are uncertain too, so every realization of the ensemble starts
from its own draw: between 5% and 15% of each state already immune, and between 0.02% and
0.1% infectious and as many exposed. Drawing them is data preparation, so it is in
`covid_inputs.initial_values`.

In [ ]:
def state_initial_values(realizations, rng):
    return covid_inputs.initial_values(
        realizations, rng, states=states, population=state_population,
        gamma=1 / INFECTIOUS_DAYS, hospitalization_prob=HOSPITALIZATION_PROB,
        hospital_days=HOSPITAL_DAYS,
    )

## 5. The simplest model: one fit and one forecast

The `state` model treats each state as one well-mixed population. Here it is fitted up to
one date and forecast from there, explained step by step. Section 7 does exactly this from
each of the 24 forecast dates, for all three models.

The date is two weeks before the region's hospital census peaked, so the four-week
forecast straddles the peak: the hardest stretch of a season to forecast.

In [ ]:
REALIZATIONS = 600  # the ensemble size
SEED = 0

peak_week = int(np.argmax(state_census.sum(axis=1)))
origin_week = peak_week - 2
origin_date = observed.census_dates[origin_week]
fit_window = TimeFrame.of(start_date=SEASON_START, duration_days=7 * origin_week)

print(f"the regional census peaked in the week of {observed.census_dates[peak_week]}")
print(f"fit {fit_window.start_date} to {fit_window.end_date} ({origin_week} weeks), "
      f"then forecast {HORIZON_WEEKS} weeks from {origin_date}")

### The model

A **RUME** (runnable modeling experiment) puts the pieces together: the compartment model,
a movement model, the scope, the time frame and the parameters. The `state` model uses
`mm.No()`: nobody moves between states.

The initializer is a placeholder: the filter is handed each realization's starting
compartments directly.

In [ ]:
state_model = SingleStrataRUME.build(
    ipm=SEIRHD(),
    mm=mm.No(),
    init=Explicit(initials=np.zeros((state_scope.nodes, 6), dtype=np.int64)),
    scope=state_scope,
    time_frame=fit_window,
    params=state_params,
)

### How the data meet the model

`Observations` says what the model's output is compared with, and how. The `ModelLink`
selects the matching model quantities: every node, the time frame grouped into seven-day
weeks, and from the compartment model the `H` compartment and the `H → D` transition. A
hospital census is a **stock**, so its week is the mean of `H`; deaths are a **flow**, so
theirs is the sum of `H → D`. The likelihood scores the observed counts against the
model's.

The source is the observations themselves, as dated values: `(weeks, nodes, 2)`, census
then deaths.

In [ ]:
def observations_for(rume, source):
    return Observations(
        source=source,
        model_link=ModelLink(
            geo=rume.scope.select.all(),
            time=rume.time_frame.select.all().group(EveryNDays(7)).agg(
                compartments="mean", events="sum"
            ),
            quantity=rume.ipm.select.by(compartments="H", events="H->D"),
        ),
        likelihood=PoissonLikelihood(),
    )


state_source = covid_inputs.date_value_source(
    state_census, state_deaths, observed.census_dates, observed.death_dates
)

### Fit: the ensemble Kalman filter

The filter runs the model forward as an ensemble of realizations, each with its own
starting state and its own parameter values drawn from the prior. At every observed week
it nudges each realization's compartments and parameters towards the data, in proportion
to how uncertain the ensemble is relative to the observation. What comes out is a fitted
ensemble: the state of the epidemic, and the estimated parameters, as of the last observed
week.

`movement_data_mode="home"` attributes every person to the node they live in, which is
where they are counted.

In [ ]:
state_fit = EnsembleKalmanFilterSimulator(
    config=PipelineConfig.from_rume(
        rume=state_model,
        num_realizations=REALIZATIONS,
        unknown_params=unknown_params,
        initial_values=state_initial_values(REALIZATIONS, np.random.default_rng(SEED)),
    ),
    observations=observations_for(state_model, state_source),
    movement_data_mode="home",
).run(rng=np.random.default_rng(SEED))

def estimated(fitted, name):
    # (realizations, days, nodes), on the log scale
    key = next(k for k in fitted.estimated_params if str(k).endswith(name))
    return fitted.estimated_params[key]


log_beta_fit = estimated(state_fit, "log_beta")
latest = np.exp(np.median(log_beta_fit[:, -1, :], axis=0))
print("estimated beta on the last fitted day:",
      ", ".join(f"{name} {value:.3f}" for name, value in zip(state_names, latest)))

### Forecast

A forecast is the fitted ensemble run forward with no more data.
`PipelineConfig.from_output` continues from where the filter stopped; `extend_duration` is
how far.

The estimated parameters keep drifting over the forecast, by new Ornstein-Uhlenbeck
dynamics centred on where the filter left them and as spread as the filter's own ensemble.
The filter's dynamics are deliberately wide, to keep it learning; carrying that width into
a forecast, where nothing is learned, would only add noise.

In [ ]:
FORECAST_DAMPING = 1 / 28
MINIMUM_FORECAST_SD = 0.01  # a collapsed ensemble must not forecast with zero spread


def forecast_dynamics(fitted):
    dynamics = {}
    for name in ("log_beta", "log_death_prob"):
        latest = estimated(fitted, name)[:, -1, :]  # (realizations, nodes)
        spread = float(np.median(np.std(latest, axis=0)))  # within a node, typical node
        dynamics[f"ipm::{name}"] = OrnsteinUhlenbeck(
            damping=FORECAST_DAMPING,
            mean=float(np.mean(latest)),
            standard_deviation=max(spread, MINIMUM_FORECAST_SD),
        )
    return dynamics


state_forecast = ForecastSimulator(
    PipelineConfig.from_output(
        state_fit,
        extend_duration=7 * HORIZON_WEEKS,
        override_dynamics=forecast_dynamics(state_fit),
    ),
    movement_data_mode="home",
).run(rng=np.random.default_rng(SEED + 1))

print(f"compartments {state_forecast.compartments.shape} "
      "(realizations, time steps, nodes, compartments)")

### Reading the forecast out

The result holds every compartment and every transition, per realization, time step and
node. The two observed quantities are read out the same way the `ModelLink` reads them:
the weekly mean of `H`, the weekly sum of `H → D`. Summarizing the realizations into
quantiles gives the forecast.

In [ ]:
H = [str(c.name) for c in SEIRHD().compartments].index("H")
H_TO_D = [str(e.name) for e in SEIRHD().events].index("H → D")
QUANTITIES = ("census", "deaths")


def weekly(result, weeks):
    # (realizations, weeks, nodes, 2): census then deaths. A movement model takes two
    # time steps a day, so the steps in a week are read off the result, not assumed.
    realizations, steps, nodes, _ = result.compartments.shape
    shape = (realizations, weeks, steps // weeks, nodes)
    census = result.compartments[..., H].reshape(shape).mean(axis=2)
    deaths = result.events[..., H_TO_D].reshape(shape).sum(axis=2)
    return np.stack([census, deaths], axis=-1)


PLOT_LEVELS = (0.025, 0.25, 0.5, 0.75, 0.975)
state_quantiles = runs.quantiles(weekly(state_forecast, HORIZON_WEEKS), PLOT_LEVELS)
state_quantiles.shape  # (weeks, states, quantities, levels)

In [ ]:
WINDOW = slice(origin_week - 8, origin_week + HORIZON_WEEKS + 4)  # the weeks to draw


def forecast_panels(forecasts):
    # One row per quantity, one column per state; `forecasts` maps a model to its
    # (weeks, states, quantities, levels) quantiles.
    rows = []
    for q, (series, dates) in enumerate((
        (state_census, observed.census_dates),
        (state_deaths, observed.death_dates),
    )):
        row = []
        for s, name in enumerate(state_names):
            row.append({
                "title": name if q == 0 else None,
                "dates": dates[WINDOW],
                "observed": series[WINDOW, s],
                "origin": dates[origin_week],
                "forecast_dates": dates[origin_week:origin_week + HORIZON_WEEKS],
                "forecasts": {
                    model: {level: values[:, s, q, i] for i, level in enumerate(PLOT_LEVELS)}
                    for model, values in forecasts.items()
                },
            })
        rows.append(row)
    return rows


fig, _ = plot_single_forecast_grid(
    forecast_panels({"state": state_quantiles}), colors=COLORS,
    ylabels=("hospital census", "weekly deaths"),
    title=f"The state model's forecast from {origin_date}",
)
plt.show()

## 6. The other two models: one change each

The comparison rests on the models differing only in what the question is about, so each
of the other two is the `state` model with one thing changed.

**`county`** splits each state into its counties: the county scope, the county
populations, and the county observations to fit. Nobody moves, so the 32 counties are 32
separate epidemics that share only their priors. Counties within a state start from their
state's level of immunity and infection, jittered by up to 25%.

In [ ]:
county_params = {**state_params, "*::population": county_population}

county_model = SingleStrataRUME.build(
    ipm=SEIRHD(),
    mm=mm.No(),
    init=Explicit(initials=np.zeros((county_scope.nodes, 6), dtype=np.int64)),
    scope=county_scope,  # was state_scope
    time_frame=fit_window,
    params=county_params,  # county populations
)

county_source = covid_inputs.date_value_source(
    observed.census, observed.deaths, observed.census_dates, observed.death_dates
)


def county_initial_values(realizations, rng):
    return covid_inputs.initial_values(
        realizations, rng, states=states, population=county_population,
        gamma=1 / INFECTIOUS_DAYS, hospitalization_prob=HOSPITALIZATION_PROB,
        hospital_days=HOSPITAL_DAYS, counties=counties,
    )

**`movement`** adds commuting to the `county` model. `mm.Pei()` moves commuters to their
work county and back every day, using the Census commuting flows from section 2.
Everything else is identical, down to the parameters, priors and initial conditions.

In [ ]:
movement_model = SingleStrataRUME.build(
    ipm=SEIRHD(),
    mm=mm.Pei(),  # was mm.No()
    init=Explicit(initials=np.zeros((county_scope.nodes, 6), dtype=np.int64)),
    scope=county_scope,
    time_frame=fit_window,
    params={**county_params, "mm::commuters": commuters},  # plus commuting flows
)

Both are fitted and forecast with the same calls as the `state` model. That takes minutes
for `county` and the better part of an hour for `movement`, whose commuting is two
movement steps a day between all 32 counties; so this notebook fits them only once, in
the rolling runs below, and section 7 ends with all three models' forecasts from the date
used above.

## 7. The rolling forecasts

The comparison repeats section 5 from every forecast date. The filter assimilates the
first eight weeks; then, week by week, the model forecasts four weeks ahead and the filter
assimilates one more week. Every forecast uses only the data before its date.

Two more pieces of bookkeeping. The county models are scored at the three states, so their
counties are summed to states **within each realization**, and only then summarized:
quantiles do not add, and the sum of 14 counties' medians is not Connecticut's median. And
every step is seeded from the run's seed afresh, so any one forecast date can be
reproduced on its own.

In [ ]:
LEVELS = HUB_QUANTILE_LEVELS  # the 23 quantiles FluSight scores: 1%, 2.5%, 5%, 10%, ..., 99%
week_dates = state_source["date"][:, 0, 0]

origins = list(range(BURN_IN_WEEKS, SEASON_WEEKS - HORIZON_WEEKS + 1))
if max_forecast_dates is not None:
    origins = origins[:max_forecast_dates]
print(f"{len(origins)} forecast dates, {week_dates[origins[0]]} to {week_dates[origins[-1]]}")

# The counties each state covers, in county order.
state_counties = [
    [i for i, county in enumerate(counties) if county[:2] == state] for state in states
]

MODELS = {  # name: (RUME, observations at its own nodes, initial values); slowest first
    "movement": (movement_model, county_source, county_initial_values),
    "county": (county_model, county_source, county_initial_values),
    "state": (state_model, state_source, state_initial_values),
}

In [ ]:
def forecast_file(reference_date, ensemble):
    # One forecast date's table, at the three states (`scored`) and, for the county
    # models, at their own counties (`model`).
    if ensemble.shape[2] == len(states):
        scales = {"scored": (runs.quantiles(ensemble, LEVELS), states)}
    else:
        by_state = np.stack(
            [ensemble[:, :, members, :].sum(axis=2) for members in state_counties], axis=2
        )
        scales = {
            "scored": (runs.quantiles(by_state, LEVELS), states),
            "model": (runs.quantiles(ensemble, LEVELS), counties),
        }
    return runs.forecast_table(reference_date, "wk inc covid hosp", scales, QUANTITIES, LEVELS)


def rolling_forecast(name, seed):
    # Burn in, then alternate: forecast four weeks, assimilate one more week.
    rume, source, initial_values = MODELS[name]
    burn_in = TimeFrame.of(start_date=SEASON_START, duration_days=7 * BURN_IN_WEEKS)
    rume = dataclasses.replace(rume, time_frame=burn_in)
    observations = observations_for(rume, source)

    fitted_blocks = []  # (posterior, prior, parameters) for each stretch the filter fits

    def assimilate(config, weeks):
        fitted = EnsembleKalmanFilterSimulator(
            config=config, observations=observations, movement_data_mode="home"
        ).run(rng=np.random.default_rng(seed))
        parameters = np.stack(
            [fitted.estimated_params[key] for key in sorted(fitted.estimated_params, key=str)],
            axis=-1,
        )
        fitted_blocks.append((fitted.posterior_values, weekly(fitted, weeks), parameters))
        return fitted

    fitted = assimilate(
        PipelineConfig.from_rume(
            rume=rume,
            num_realizations=REALIZATIONS,
            unknown_params=unknown_params,
            initial_values=initial_values(REALIZATIONS, np.random.default_rng(seed)),
        ),
        weeks=BURN_IN_WEEKS,
    )

    forecasts = []
    for position, origin in enumerate(origins):
        forecast = ForecastSimulator(
            PipelineConfig.from_output(
                fitted,
                extend_duration=7 * HORIZON_WEEKS,
                override_dynamics=forecast_dynamics(fitted),
            ),
            movement_data_mode="home",
        ).run(rng=np.random.default_rng(seed + 1 + position))
        forecasts.append(forecast_file(week_dates[origin], weekly(forecast, HORIZON_WEEKS)))

        if origin != origins[-1]:
            fitted = assimilate(PipelineConfig.from_output(fitted, 7), weeks=1)

    # The fit, week by week, and the estimated parameters, day by day.
    posterior, prior, parameters = (
        np.concatenate(blocks, axis=1) for blocks in zip(*fitted_blocks)
    )
    nodes = list(rume.scope.node_ids)
    fit = runs.fit_table(
        posterior, prior, week_dates[:posterior.shape[1]], nodes, QUANTITIES, LEVELS
    )
    days = [np.datetime64(SEASON_START + timedelta(days=d)) for d in range(parameters.shape[1])]
    names = sorted(str(key) for key in fitted.estimated_params)
    params = runs.params_table(parameters, np.array(days), nodes, names, LEVELS)
    return forecasts, fit, params

Each run is one model at one seed, and is written to `runs_dir/seed<N>/<model>/`. The
manuscript uses two seeds, so that every comparison can be checked for agreement between
them. A run already on disk with the same settings is reused; a missing one is fitted. The
movement model takes about an hour and a half, so fitting all six, four at a time, takes
about as long.

In [ ]:
SEEDS = (0, 1)
PARALLEL_RUNS = 4  # the demo waits for its slowest model; four starts every slow run at once


def run_folder(name, seed):
    return Path(runs_dir) / f"seed{seed}" / name


def run_settings(seed):
    return {
        "season_start": str(SEASON_START),
        "season_weeks": origins[-1] + HORIZON_WEEKS,
        "burn_in_weeks": BURN_IN_WEEKS,
        "horizon_weeks": HORIZON_WEEKS,
        "num_realizations": REALIZATIONS,
        "seed": seed,
    }


def fit_and_forecast(name, seed):
    # One run: rolling forecasts for one model at one seed, written to disk.
    started = time.monotonic()
    forecasts, fit, params = rolling_forecast(name, seed)
    minutes = (time.monotonic() - started) / 60
    runs.write_run(
        run_folder(name, seed), forecasts=forecasts, fit=fit, params=params,
        info={"model": name, **run_settings(seed), "elapsed_minutes": round(minutes, 1)},
    )
    return f"{name}, seed {seed}: {minutes:.0f} min"


missing = []
for name in MODELS:  # slowest first
    for seed in SEEDS:
        if not runs.run_on_disk(run_folder(name, seed), run_settings(seed)):
            missing.append((name, seed))

print(f"{len(MODELS) * len(SEEDS) - len(missing)} runs on disk, {len(missing)} to fit")

# Each run in its own process. joblib can send functions defined in this notebook to
# those processes; the standard library cannot.
finished = Parallel(n_jobs=PARALLEL_RUNS, verbose=10)(
    delayed(fit_and_forecast)(name, seed) for name, seed in missing
)
for line in finished:
    print(line)

### Three models, one forecast date

The forecasts the three models made from the date of section 5, read back from the runs
at seed 0. The `state` model's differs a little from section 5's, because the rolling run
reached that date one assimilated week at a time.

In [ ]:
from_runs = {}
for name in MODELS:
    path = run_folder(name, 0) / "forecasts" / f"{origin_date}.csv"
    if not path.exists():  # a quick test run stops before this date
        continue
    table = pd.read_csv(path, dtype={"location": str})
    table = table[(table["scale"] == "scored") & table["quantile"].isin(PLOT_LEVELS)]
    # Sorted by horizon, location, quantity, then quantile, so it reshapes directly.
    from_runs[name] = table["value"].to_numpy().reshape(
        HORIZON_WEEKS, len(states), len(QUANTITIES), len(PLOT_LEVELS)
    )

if from_runs:
    fig, _ = plot_single_forecast_grid(
        forecast_panels(from_runs), colors=COLORS,
        ylabels=("hospital census", "weekly deaths"),
        title=f"Three models, one forecast date: {origin_date}",
    )
    plt.show()
else:
    print(f"no run reaches {origin_date}")